In [1]:
# 1138-01  Use 2024 computer rates

In [1]:
import geopandas as gpd
from shapely.geometry import Point, Polygon, LineString
import numpy as np
import pandas as pd
import glob
import re
import matplotlib.pyplot as plt
import os
import pyvista as pv
from pathlib import Path

pd.set_option('display.max_columns', None)



In [2]:
spath = r"C:\Users\cakyol\gpkgs\**.gpkg"
files = [x.replace('\\','/') for x in glob.glob(spath, recursive=True)]

#spath = r"C:\scipts\1138-01\Sonar\PPG2\1993\cynthia\*.gpkg"
#files = files + [x.replace('\\','/') for x in glob.glob(spath, recursive=True)]

print(len(files))
df = pd.DataFrame({'path':files})
df['fname'] = df['path'].apply(lambda x: x.split('/')[-1])
df['orient'] = 'v'
df['orient'] = df['orient'].where(df.fname.apply( lambda x:'ft' not in x), other='h')

# Drop Bad row
df.drop(index=[14], inplace=True)

## well
import re

df['Well_No'] = None

for index, row in df.iterrows():

    fname = row['fname']
    fname_upper = fname.upper()

    # -------------------------
    # HORIZONTAL FILES
    # -------------------------
    if row['orient'] == 'h':

        m = re.search(r'\d{4}', fname)
        if m:
            df.at[index, 'z'] = int(m.group())

        df.at[index, 'Well_No'] = 'all'


    # -------------------------
    # VERTICAL FILES
    # -------------------------
    else:

        # extract azimuths
        m = re.search(r'\d+-\d+', fname)
        if m:
            a0, a1 = m.group().split('-')
            df.at[index, 'azi0'] = int(a0)
            df.at[index, 'azi1'] = int(a1)

        # assign well based on filename
        if 'ALL' in fname_upper:
            df.at[index, 'Well_No'] = 'all'

        elif 'PPG4' in fname_upper:
            df.at[index, 'Well_No'] = 'PPG4'

        elif 'PPG2' in fname_upper:
            df.at[index, 'Well_No'] = 'PPG2'

        else:
            df.at[index, 'Well_No'] = 'UNKNOWN'

27


In [3]:
## dX, dY
# ppg 2 
2624245.599498,643042.320207

# ppg 4
2624685.606284,642633.314476

#horizontal offsets
#ppg2 dy:155, dx:-290
#ppg4 dy:-120, dx:180

(2624685.606284, 642633.314476)

In [4]:
# USER INPUTS

CRS_OUT = "EPSG:3452"

# Main 0-reference point for the cavern
CAVERN_CENTER = {
    "X0": 2624465.602891,
    "Y0": 642837.817341
}
# Wellhead offsets from cavern center
# dx = East-West difference
# dy = North-South difference
WELL_OFFSETS_FROM_CAVERN_CENTER = {
    "PPG2": {
        "dx": -300,
        "dy":  155
    },
    "PPG4": {
        "dx":  180,
        "dy": -120
    }
}

# Optional: rebuild WELLHEADS automatically from center + offsets
WELLHEADS = {
    well: {
        "Xwh": CAVERN_CENTER["X0"] + off["dx"],
        "Ywh": CAVERN_CENTER["Y0"] + off["dy"]
    }
    for well, off in WELL_OFFSETS_FROM_CAVERN_CENTER.items()
}

SCALE_LOCAL_TO_MAP = 1.0
STEP = 10.0
VERT_AZ_MODE = "mean"
VERTICAL_X_IS_YLOCAL = True


WELLS = {
    "PPG2": {"X0": WELLHEADS["PPG2"]["Xwh"], "Y0": WELLHEADS["PPG2"]["Ywh"], "color": "mediumpurple"},
    "PPG4": {"X0": WELLHEADS["PPG4"]["Xwh"], "Y0": WELLHEADS["PPG4"]["Ywh"], "color": "sienna"},
}


In [5]:
# Helper functions
def circular_mean_deg(a, b):
    a = np.deg2rad(a); b = np.deg2rad(b)
    x = np.cos(a) + np.cos(b)
    y = np.sin(a) + np.sin(b)
    return (np.rad2deg(np.arctan2(y, x)) + 360) % 360

def get_vertical_azimuths(row):
    a0, a1 = row.get("azi0", np.nan), row.get("azi1", np.nan)
    if pd.isna(a0) and pd.isna(a1):
        return []
    if VERT_AZ_MODE == "azi0":
        return [float(a0)]
    if VERT_AZ_MODE == "azi1":
        return [float(a1)]
    if VERT_AZ_MODE == "mean":
        return [float(circular_mean_deg(float(a0), float(a1)))]
    if VERT_AZ_MODE == "both":
        out = []
        if not pd.isna(a0): out.append(float(a0))
        if not pd.isna(a1): out.append(float(a1))
        return out
    raise ValueError("VERT_AZ_MODE must be one of: azi0, azi1, mean, both")

def sample_geom_xy(geom, step=STEP):
    if geom is None or geom.is_empty:
        return np.empty((0, 2), dtype=float)

    if isinstance(geom, Polygon):
        line = geom.exterior
    elif isinstance(geom, LineString):
        line = geom
    else:
        b = geom.boundary
        if isinstance(b, LineString):
            line = b
        else:
            return np.empty((0, 2), dtype=float)

    L = line.length
    if L <= 0:
        return np.empty((0, 2), dtype=float)

    n = max(2, int(np.ceil(L / step)))
    dists = np.linspace(0, L, n)
    pts = [line.interpolate(d) for d in dists]
    return np.array([[p.x, p.y] for p in pts], dtype=float)

def cavern_center_from_wells():
    centers = {
        well: (
            CAVERN_CENTER["X0"],
            CAVERN_CENTER["Y0"]
        )
        for well in WELL_OFFSETS_FROM_CAVERN_CENTER.keys()
    }
    return (CAVERN_CENTER["X0"], CAVERN_CENTER["Y0"]), centers

def wellhead_from_center(well):
    Xw = CAVERN_CENTER["X0"] + WELL_OFFSETS_FROM_CAVERN_CENTER[well]["dx"]
    Yw = CAVERN_CENTER["Y0"] + WELL_OFFSETS_FROM_CAVERN_CENTER[well]["dy"]
    return Xw, Yw
    
def section_origin_from_well(well, az_deg, offset):
    Xw, Yw = wellhead_from_center(well)
    az = np.deg2rad(float(az_deg))
    X0 = Xw - offset * np.sin(az)
    Y0 = Yw - offset * np.cos(az)
    return X0, Y0

Xsec_from_ppg2, Ysec_from_ppg2 = section_origin_from_well("PPG2", 300, -330)
Xsec_from_ppg4, Ysec_from_ppg4 = section_origin_from_well("PPG4", 300, 210)

print("300-120 origin from PPG2:", Xsec_from_ppg2, Ysec_from_ppg2)
print("300-120 origin from PPG4:", Xsec_from_ppg4, Ysec_from_ppg4)
print("Difference:", Xsec_from_ppg2 - Xsec_from_ppg4, Ysec_from_ppg4 - Ysec_from_ppg4)

Xsec_300120 = 0.5 * (Xsec_from_ppg2 + Xsec_from_ppg4)
Ysec_300120 = 0.5 * (Ysec_from_ppg2 + Ysec_from_ppg4)

print("Using 300-120 section origin:", Xsec_300120, Ysec_300120)
print("Difference:", Xsec_from_ppg2 - Xsec_from_ppg4, Ysec_from_ppg2 - Ysec_from_ppg4)

300-120 origin from PPG2: 2623879.8145077513 643157.817341
300-120 origin from PPG4: 2624827.468225795 642612.817341
Difference: -947.6537180435844 0.0
Using 300-120 section origin: 2624353.6413667733 642885.317341
Difference: -947.6537180435844 545.0


In [6]:

# Build referenced point cloud + metadata



(Xcav, Ycav), centers = cavern_center_from_wells()
print("Cavern center from wells (per-well):", centers)
print("Using cavern center (average):", (Xcav, Ycav))


Xsec = 0.5 * (WELLHEADS["PPG2"]["Xwh"] + WELLHEADS["PPG4"]["Xwh"])
Ysec = 0.5 * (WELLHEADS["PPG2"]["Ywh"] + WELLHEADS["PPG4"]["Ywh"])


# Fit cavern center for 300-120_all
az_300120 = np.deg2rad(120.0)

# distances along section axis from cavern center
s_ppg2 = -330.0
s_ppg4 = 290.0

Xcav2 = WELLHEADS["PPG2"]["Xwh"] - s_ppg2 * np.sin(az_300120)
Ycav2 = WELLHEADS["PPG2"]["Ywh"] - s_ppg2 * np.cos(az_300120)

Xcav4 = WELLHEADS["PPG4"]["Xwh"] - s_ppg4 * np.sin(az_300120)
Ycav4 = WELLHEADS["PPG4"]["Ywh"] - s_ppg4 * np.cos(az_300120)

Xcav_fit = 0.5 * (Xcav2 + Xcav4)
Ycav_fit = 0.5 * (Ycav2 + Ycav4)

print("Fitted cavern center for 300-120:", (Xcav_fit, Ycav_fit))

records = []

for _, row in df.iterrows():

    orient = row["orient"]
    fname  = row["fname"]
    path   = row["path"]
    wellno = str(row.get("Well_No", "all")).lower()

    gdf = gpd.read_file(path)
    if gdf.empty:
        continue


    # HORIZONTAL: cavern-centered

    if orient == "h":

        depth = row.get("z", np.nan)
        if pd.isna(depth):
            continue

        for geom in gdf.geometry:
            xy = sample_geom_xy(geom, step=STEP)
            if xy.size == 0:
                continue

            x_local = xy[:, 0]
            y_local = xy[:, 1]

            X = Xcav + x_local
            Y = Ycav + y_local
            Z = np.full_like(X, -float(depth), dtype=float)

            for xi, yi, zi in zip(X, Y, Z):
                records.append({
                    "fname": fname,
                    "path": path,
                    "orient": "h",
                    "depth": float(depth),
                    "azi0": np.nan,
                    "azi1": np.nan,
                    "azi": np.nan,
                    "Well_No": wellno,
                    "geometry": Point(float(xi), float(yi), float(zi)),
                })


    # VERTICAL

    elif orient == "v":

        az_list = get_vertical_azimuths(row)
        if not az_list:
            continue

        a0 = row.get("azi0", np.nan)
        a1 = row.get("azi1", np.nan)

        # choose anchor
        if wellno == "ppg2":
            X0 = WELLHEADS["PPG2"]["Xwh"]
            Y0 = WELLHEADS["PPG2"]["Ywh"]

        elif wellno == "ppg4":
            X0 = WELLHEADS["PPG4"]["Xwh"]
            Y0 = WELLHEADS["PPG4"]["Ywh"]

        elif wellno == "all":
            if "300-120" in fname:
                X0 = Xcav_fit
                Y0 = Ycav_fit
            else:
                X0 = Xsec
                Y0 = Ysec

        else:
            print(f"Skipping {fname}: unknown Well_No = {wellno}")
            continue

        for az_deg in az_list:

#ROTATING WITH RELATIVE AZIMUTH************************************************
#300-120_all
            if "300-120" in fname:
                az_use = 120.0
#277-97_PPG2
            elif "277-97" in fname and wellno == "ppg2":
                # rotate relative to original azimuth, keep wellhead fixed
                az_use = (float(az_deg) + 90.0) % 360
                print("ROTATING 277-97_PPG2:", fname, "old:", az_deg, "new:", az_use)
#270-90_PPG4_2
            elif fname == "270-90_PPG4_2.gpkg" and wellno == "ppg4":
                az_use = (float(az_deg) - 90.0) % 360
                print("ROTATING:", fname, "old:", az_deg, "new:", az_use)
#270-90_PPG4_1
            elif fname == "270-90_PPG4_1.gpkg" and wellno == "ppg4":
                az_use = (float(az_deg) - 90.0) % 360
                print("ROTATING:", fname, "old:", az_deg, "new:", az_use)
#315-135_PPG4
            elif fname == "315-135_PPG4.gpkg" and wellno == "ppg4":
                az_use = (float(az_deg) - 90.0) % 360
                print("ROTATING:", fname, "old:", az_deg, "new:", az_use)
#315-135_PPG2
            elif fname == "315-135_PPG2.gpkg" and wellno == "ppg2":
                az_use = (float(az_deg) - 90.0) % 360
                print("ROTATING:", fname, "old:", az_deg, "new:", az_use)
#225-45_PPG2
            elif fname == "225-45_PPG2.gpkg" and wellno == "ppg2":
                az_use = (float(az_deg) + 90.0) % 360
                print("ROTATING:", fname, "old:", az_deg, "new:", az_use)

#225-45_PPG4
            elif fname == "225-45_PPG4.gpkg" and wellno == "ppg4":
                az_use = (float(az_deg) - 90.0) % 360
                print("ROTATING:", fname, "old:", az_deg, "new:", az_use)
#0-180_PPG4
            elif fname == "0-180_PPG4.gpkg" and wellno == "ppg4":
                az_use = (float(az_deg) - 90.0) % 360
                print("ROTATING:", fname, "old:", az_deg, "new:", az_use)
#0-180_PPG4
            elif fname == "0-180_PPG2.gpkg" and wellno == "ppg2":
                az_use = (float(az_deg) +90.0) % 360
                print("ROTATING:", fname, "old:", az_deg, "new:", az_use)

#0-180_PPG4
            elif fname == "270-90_PPG2.gpkg" and wellno == "ppg2":
                az_use = (float(az_deg) - 90.0) % 360
                print("ROTATING:", fname, "old:", az_deg, "new:", az_use)
            else:
                az_use = float(az_deg)

            az = np.deg2rad(az_use)
#*****************************************************************************
#SHIFTS

    
            x_shift = 0.0
            y_shift = 0.0

            for geom in gdf.geometry:
                xy = sample_geom_xy(geom, step=STEP)
                if xy.size == 0:
                    continue

                if VERTICAL_X_IS_YLOCAL:
                    y_local = xy[:, 0]
                    z_local = xy[:, 1]
                else:
                    y_local = xy[:, 1]
                    z_local = xy[:, 0]

                dX = y_local * np.sin(az)
                dY = y_local * np.cos(az)

                X = X0 + dX + x_shift
                Y = Y0 + dY + y_shift
                Z = z_local.astype(float)

                for xi, yi, zi in zip(X, Y, Z):
                    records.append({
                        "fname": fname,
                        "path": path,
                        "orient": "v",
                        "depth": np.nan,
                        "azi0": float(a0) if not pd.isna(a0) else np.nan,
                        "azi1": float(a1) if not pd.isna(a1) else np.nan,
                        "az_used": az_use,
                        "azi" : az_use,
                        "Well_No": wellno,
                        "geometry": Point(float(xi), float(yi), float(zi)),
                    })

gdf_out = gpd.GeoDataFrame(records, geometry="geometry", crs=CRS_OUT)
print("Total referenced points:", len(gdf_out))

Cavern center from wells (per-well): {'PPG2': (2624465.602891, 642837.817341), 'PPG4': (2624465.602891, 642837.817341)}
Using cavern center (average): (2624465.602891, 642837.817341)
Fitted cavern center for 300-120: (2624422.923399076, 642845.317341)
ROTATING: 0-180_PPG2.gpkg old: 90.0 new: 180.0
ROTATING: 0-180_PPG4.gpkg old: 90.0 new: 0.0
ROTATING: 225-45_PPG2.gpkg old: 135.0 new: 225.0
ROTATING: 225-45_PPG4.gpkg old: 135.0 new: 45.0
ROTATING: 270-90_PPG2.gpkg old: 180.0 new: 90.0
ROTATING: 270-90_PPG4_1.gpkg old: 180.0 new: 90.0
ROTATING: 270-90_PPG4_2.gpkg old: 180.0 new: 90.0
ROTATING 277-97_PPG2: 277-97_PPG2.gpkg old: 19.17900802581073 new: 109.17900802581073
ROTATING: 315-135_PPG2.gpkg old: 225.0 new: 135.0
ROTATING: 315-135_PPG4.gpkg old: 225.0 new: 135.0
Total referenced points: 6548


In [7]:

def shift_point_3d(p, dx=0.0, dy=0.0, dz=0.0):
    return Point(p.x + dx, p.y + dy, p.z + dz)

def rotate_point_xy_around_anchor(p, angle_deg, x0, y0):
    """
    Rotate a 3D point in the XY plane around anchor (x0, y0).
    Z stays unchanged.
    Positive angle = counterclockwise.
    """
    ang = np.deg2rad(angle_deg)

    dx = p.x - x0
    dy = p.y - y0

    xr = dx * np.cos(ang) - dy * np.sin(ang)
    yr = dx * np.sin(ang) + dy * np.cos(ang)

    return Point(x0 + xr, y0 + yr, p.z)

def transform_gdf_points(
    gdf,
    mask,
    angle_deg=0.0,
    anchor_x=None,
    anchor_y=None,
    dx=0.0,
    dy=0.0,
    dz=0.0
):
    gdf2 = gdf.copy()

    def _transform(p):
        p2 = p

        if angle_deg != 0.0:
            if anchor_x is None or anchor_y is None:
                raise ValueError("anchor_x and anchor_y are required for rotation.")
            p2 = rotate_point_xy_around_anchor(p2, angle_deg, anchor_x, anchor_y)

        if dx != 0.0 or dy != 0.0 or dz != 0.0:
            p2 = shift_point_3d(p2, dx=dx, dy=dy, dz=dz)

        return p2

    gdf2.loc[mask, "geometry"] = gdf2.loc[mask, "geometry"].apply(_transform)
    return gdf2

In [8]:
import pyvista as pv
import numpy as np
import matplotlib.pyplot as plt

plotter = pv.Plotter()

files = gdf_out["fname"].unique()
colors = plt.cm.tab20(np.linspace(0, 1, len(files)))

for i, fname in enumerate(files):

    sub = gdf_out[gdf_out["fname"] == fname]

    coords = np.array([
        (g.x, g.y, g.z)
        for g in sub.geometry
    ])

    cloud = pv.PolyData(coords)

    # Highlight important sections
    if "270-90" in fname:
        plotter.add_mesh(
            cloud,
            color="red",
            render_points_as_spheres=True,
            point_size=12,
            label=fname
        )

    else:
        plotter.add_mesh(
            cloud,
            color=colors[i][:3],
            render_points_as_spheres=True,
            point_size=5,
            opacity=0.5,
            label=fname
        )

pv.global_theme.font.size = 58

plotter.add_legend(
    loc="upper right",
    size=(0.15, 0.15),
    bcolor="white"
)

plotter.show()

Widget(value='<iframe src="http://localhost:59490/index.html?ui=P_0x24dda1dae50_0&reconnect=auto" class="pyvis…

In [9]:
# EDIT ONE VERTICAL SECTION AFTER BUILD

#Bullk shift of 300-120_all.gpkg

mask = (
    (gdf_out["orient"] == "v") &
    (gdf_out["fname"].str.contains("300-120", case=False, na=False))
)

gdf_edit = transform_gdf_points(
    gdf_out,
    mask=mask,
    angle_deg=0.0,         # try 2, -2, 5, etc.
    anchor_x=Xcav_fit,     # or Xsec / PPG2 / PPG4 wellhead
    anchor_y=Ycav_fit,
    dx=30.0,
    dy=-28.0,
    dz=0.0
)

#Rotating 315-135_PPG4 (includes 1981 sonar)
# second fix
mask = (
    (gdf_edit["orient"] == "v") &
    (gdf_edit["fname"].str.contains("277-97_PPG2", case=False, na=False)) &
    (gdf_edit["Well_No"] == "ppg2")
)

gdf_edit = transform_gdf_points(
    gdf_edit,
    mask=mask,
    angle_deg=0.0,
    anchor_x=WELLHEADS["PPG2"]["Xwh"],
    anchor_y=WELLHEADS["PPG2"]["Ywh"],
    dx=280.0,
    dy=-125.0,
    dz=0.0
)
# -------------------------------
# SHIFT 2700ft HORIZONTAL TO 2900ft
# -------------------------------

mask = (
    (gdf_edit["orient"] == "h") &
    (gdf_edit["fname"].str.contains("2700", case=False, na=False))
)

gdf_edit = transform_gdf_points(
    gdf_edit,
    mask=mask,
    angle_deg=0.0,   # no rotation
    anchor_x=0,
    anchor_y=0,
    dx=0.0,
    dy=0.0,
    dz=-50.0        # move from 2700 to 2900
)


In [10]:

gdf_edit.to_file(r"C:\Users\cakyol\gpkgs\gdf_edit_export.gpkg", driver="GPKG")

plotter = pv.Plotter()

files = gdf_edit["fname"].unique()
colors = plt.cm.tab20(np.linspace(0, 1, len(files)))

for i, fname in enumerate(files):

    sub = gdf_edit[gdf_edit["fname"] == fname]

    coords = np.array([
        (g.x, g.y, g.z)
        for g in sub.geometry
    ])

    cloud = pv.PolyData(coords)

    if "225-45" in fname:
        plotter.add_mesh(
            cloud,
            color="red",
            render_points_as_spheres=True,
            point_size=12,
            label=fname
        )
    else:
        plotter.add_mesh(
            cloud,
            color=colors[i][:3],
            render_points_as_spheres=True,
            point_size=5,
            opacity=0.5,
            label=fname
        )

pv.global_theme.font.size = 58

plotter.add_legend(
    loc="upper right",
    size=(0.15, 0.15),
    bcolor="white"
)

plotter.show()

Widget(value='<iframe src="http://localhost:59490/index.html?ui=P_0x24dd85ddd90_1&reconnect=auto" class="pyvis…

In [11]:
gdf_edit

,fname,path,orient,depth,azi0,azi1,az_used,azi,Well_No,geometry
0,0-180_PPG2.gpkg,C:/Users/cakyol/gpkgs/0-180_PPG2.gpkg,v,NaN,0.0,180.0,180.0,180.0,ppg2,POINT Z (2624165.603 642994.954 -2430.456)
1,0-180_PPG2.gpkg,C:/Users/cakyol/gpkgs/0-180_PPG2.gpkg,v,NaN,0.0,180.0,180.0,180.0,ppg2,POINT Z (2624165.603 642984.923 -2430.486)
2,0-180_PPG2.gpkg,C:/Users/cakyol/gpkgs/0-180_PPG2.gpkg,v,NaN,0.0,180.0,180.0,180.0,ppg2,POINT Z (2624165.603 642986.192 -2439.265)
3,0-180_PPG2.gpkg,C:/Users/cakyol/gpkgs/0-180_PPG2.gpkg,v,NaN,0.0,180.0,180.0,180.0,ppg2,POINT Z (2624165.603 642984.017 -2448.074)
4,0-180_PPG2.gpkg,C:/Users/cakyol/gpkgs/0-180_PPG2.gpkg,v,NaN,0.0,180.0,180.0,180.0,ppg2,POINT Z (2624165.603 642974.145 -2448.778)
...,...,...,...,...,...,...,...,...,...,...
6543,315-135_PPG4.gpkg,C:/Users/cakyol/gpkgs/315-135_PPG4.gpkg,v,NaN,315.0,135.0,135.0,135.0,ppg4,POINT Z (2624632.796 642730.624 -2645.626)
6544,315-135_PPG4.gpkg,C:/Users/cakyol/gpkgs/315-135_PPG4.gpkg,v,NaN,315.0,135.0,135.0,135.0,ppg4,POINT Z (2624639.865 642723.555 -2645.29)
6545,315-135_PPG4.gpkg,C:/Users/cakyol/gpkgs/315-135_PPG4.gpkg,v,NaN,315.0,135.0,135.0,135.0,ppg4,POINT Z (2624643.575 642719.846 -2639.995)
6546,315-135_PPG4.gpkg,C:/Users/cakyol/gpkgs/315-135_PPG4.gpkg,v,NaN,315.0,135.0,135.0,135.0,ppg4,POINT Z (2624643.741 642719.679 -2629.994)


In [55]:
import geopandas as gpd
from shapely.geometry import Point
from scipy.interpolate import interp1d
from scipy.ndimage import gaussian_filter

# --------------------------------------------------
# 1. Anchors
# --------------------------------------------------
anchors = {
    'ppg2': (WELLHEADS['PPG2']['Xwh'], WELLHEADS['PPG2']['Ywh']),
    'ppg4': (WELLHEADS['PPG4']['Xwh'], WELLHEADS['PPG4']['Ywh']),
    'all' : (Xcav_fit, Ycav_fit),
}

# --------------------------------------------------
# 2. Attach anchor coordinates
# --------------------------------------------------
gdf_edit = gdf_edit.copy()
gdf_edit['Well_No'] = gdf_edit['Well_No'].str.lower()
gdf_edit['X0'] = gdf_edit['Well_No'].map(lambda k: anchors[k][0])
gdf_edit['Y0'] = gdf_edit['Well_No'].map(lambda k: anchors[k][1])

# --------------------------------------------------
# 3. Compute azimuth / depth / radius for vertical points
# --------------------------------------------------
gdf_v = gdf_edit[gdf_edit['orient'] == 'v'].copy()
gdf_v['azi']   = gdf_v.apply(lambda row: (np.degrees(np.arctan2(row.geometry.x - row['X0'], row.geometry.y - row['Y0'])) + 360) % 360, axis=1)
gdf_v['depth'] = gdf_v.geometry.apply(lambda p: p.z)
gdf_v['r']     = gdf_v.apply(lambda row: np.sqrt((row.geometry.x - row['X0'])**2 + (row.geometry.y - row['Y0'])**2), axis=1)

df = gdf_v[['azi', 'r', 'depth', 'fname', 'Well_No', 'X0', 'Y0']].reset_index(drop=True)

# --------------------------------------------------
# 4. Synthetic fill settings
# --------------------------------------------------
dz                 = 2
dazi               = 2
max_gap            = 200

min_points_per_bin = 1

syn_list   = []
depth_bins = np.arange(df.depth.min(), df.depth.max() + dz, dz)

# --------------------------------------------------
# 5. Interpolate separately by anchor group
# --------------------------------------------------
for well_no, g in df.groupby('Well_No'):
    if well_no == "all":
        continue

    for d in depth_bins:
        t = g[(g.depth >= d) & (g.depth < d + dz)].copy()
        if t.empty or len(t) < min_points_per_bin:
            continue

        t['azi_grid'] = np.round(t['azi'] / dazi) * dazi % 360
        t = (t.groupby('azi_grid', as_index=False)
              .agg({'r': 'mean', 'depth': 'mean', 'X0': 'mean', 'Y0': 'mean'})
              .rename(columns={'azi_grid': 'azi'})
              .sort_values('azi')
              .reset_index(drop=True))

        if len(t) < min_points_per_bin:
            continue

        real_azi = t['azi'].values
        azi_ext  = np.r_[real_azi - 360, real_azi, real_azi + 360]
        r_ext    = np.r_[t['r'].values,  t['r'].values,  t['r'].values]
        X0_ext   = np.r_[t['X0'].values, t['X0'].values, t['X0'].values]
        Y0_ext   = np.r_[t['Y0'].values, t['Y0'].values, t['Y0'].values]

        full_azi    = np.arange(0, 360, dazi)
        missazi     = np.setdiff1d(full_azi, real_azi)
        real_sorted = np.sort(real_azi)
        real_wrap   = np.r_[real_sorted, real_sorted[0] + 360]

        valid_azi = []
        for a in missazi:
            for i in range(len(real_sorted)):
                a0, a1 = real_wrap[i], real_wrap[i + 1]
                if a1 - a0 <= max_gap:
                    aa = a if a >= a0 else a + 360
                    if a0 < aa < a1:
                        valid_azi.append(a)
                        break

        if len(valid_azi) == 0:
            continue

        valid_azi = np.array(valid_azi)
        r_syn     = np.interp(valid_azi, azi_ext, r_ext)
        X0_syn    = np.interp(valid_azi, azi_ext, X0_ext)
        Y0_syn    = np.interp(valid_azi, azi_ext, Y0_ext)

        syn_list.append(pd.DataFrame({
            'azi':     valid_azi,
            'depth':   t['depth'].mean(),
            'r':       r_syn,
            'X0':      X0_syn,
            'Y0':      Y0_syn,
            'Well_No': well_no,
            'is_syn':  True,
        }))

syn = pd.concat(syn_list, ignore_index=True) if syn_list else pd.DataFrame()

# --------------------------------------------------
# 6. Reconstruct XYZ + r filter
# --------------------------------------------------
syn['x'] = syn['X0'] + syn['r'] * np.sin(np.deg2rad(syn['azi']))
syn['y'] = syn['Y0'] + syn['r'] * np.cos(np.deg2rad(syn['azi']))
syn['z'] = syn['depth']

R_MAX_PLOT = 300
for wn, wh in WELLHEADS.items():
    mask = syn["Well_No"] == wn.lower()
    syn  = syn[~mask | (syn.loc[mask, "r"] <= R_MAX_PLOT)]

print(f"syn after r filter: {len(syn)} points")

# --------------------------------------------------
# 6a. Mirror helpers  (reused in steps 6c and 7)
# --------------------------------------------------
def find_mirror_axis(covered_azi):
    """Circular mean of covered azimuths -- bisects the covered arc."""
    rad = np.deg2rad(covered_azi)
    return (np.degrees(np.arctan2(np.sin(rad).mean(), np.cos(rad).mean())) + 360) % 360

def mirror_fill(covered_azi, covered_r, missing_azi, mirror_axis_deg):
    """
    Reflect each missing azimuth about mirror_axis_deg and
    interpolate its radius from the covered profile.
    """
    reflected = (2 * mirror_axis_deg - missing_azi) % 360
    sort_idx  = np.argsort(covered_azi)
    azi_s     = covered_azi[sort_idx]
    r_s       = covered_r[sort_idx]
    azi_ext   = np.r_[azi_s - 360, azi_s, azi_s + 360]
    r_ext     = np.r_[r_s, r_s, r_s]
    return np.interp(reflected, azi_ext, r_ext)

# --------------------------------------------------
# 6b. Zone definitions
#     Zone 1 : Z_ZONE1_BOT <= depth <= Z_ZONE1_TOP  vertical-based synthetic
#     Zone 2 : Z_ZONE2_BOT <= depth <  Z_ZONE1_BOT  horizontal-based rings
#     Zone 3 : depth        <  Z_ZONE2_BOT           vertical-based synthetic
# --------------------------------------------------
Z_ZONE1_TOP = -2400   # shallowest depth of interest
Z_ZONE1_BOT = -2800   # zone 1 / zone 2 boundary
Z_ZONE2_BOT = -2910   # zone 2 / zone 3 boundary

# Restrict vertical-based synthetic to zones 1 and 3
syn_vert = syn[
    ((syn["depth"] >= Z_ZONE1_BOT) & (syn["depth"] <= Z_ZONE1_TOP)) |
    (syn["depth"] <  Z_ZONE2_BOT)
].copy()
print(f"syn_vert (zones 1+3): {len(syn_vert)} points")

# --------------------------------------------------
# 6c. Zone 2: horizontal-based rings with vertical interpolation
# --------------------------------------------------
horiz_real = gdf_edit[gdf_edit["orient"] == "h"].copy()

def build_horiz_rings(horiz_gdf, depth_min, depth_max, dz, dazi, anchor_x, anchor_y,
                      n_between=2):
    """
    For each pair of adjacent real horizontal sections create n_between
    synthetic rings at evenly-spaced intermediate depths.

    Each synthetic ring's radius profile is linearly interpolated between
    the two bounding sections so the rings float independently between them —
    no connecting geometry, no dense stack.
    """
    pts = horiz_gdf.copy()
    pts["_z"]   = pts.geometry.apply(lambda g: g.z)
    pts["_azi"] = pts.geometry.apply(
        lambda g: (np.degrees(np.arctan2(g.x - anchor_x, g.y - anchor_y)) + 360) % 360
    )
    pts["_r"] = pts.geometry.apply(
        lambda g: np.sqrt((g.x - anchor_x)**2 + (g.y - anchor_y)**2)
    )
    pts["_d_bin"]   = (pts["_z"]   / dz).round()   * dz
    pts["_azi_bin"] = (pts["_azi"] / dazi).round() * dazi % 360

    pts = pts[(pts["_d_bin"] >= depth_min) & (pts["_d_bin"] <= depth_max)]
    if pts.empty:
        print("  No horizontal data found in zone 2 depth range.")
        return pd.DataFrame()

    h = (pts.groupby(["_d_bin", "_azi_bin"], as_index=False)
           .agg({"_r": "mean"})
           .rename(columns={"_d_bin": "depth", "_azi_bin": "azi", "_r": "r"}))

    h_depths = np.sort(h["depth"].unique())
    all_azi  = np.arange(0, 360, dazi)

    print(f"  Horizontal slices in zone 2: z = {h_depths}")

    # Build a complete (all-azi) ring for every real horizontal slice
    slice_rings = {}
    for d_ref in h_depths:
        s      = h[h["depth"] == d_ref].set_index("azi")["r"].reindex(all_azi)
        r_vals = s.values
        pres   = ~np.isnan(r_vals)
        if pres.sum() == 0:
            continue
        cov_a, cov_r = all_azi[pres], r_vals[pres]
        miss_a = all_azi[~pres]
        if len(miss_a) > 0:
            ax    = find_mirror_axis(cov_a)
            cov_a = np.r_[cov_a, miss_a]
            cov_r = np.r_[cov_r, mirror_fill(all_azi[pres], r_vals[pres], miss_a, ax)]
        # Re-index onto all_azi so every ring shares the same grid
        sort_idx  = np.argsort(cov_a)
        slice_rings[d_ref] = np.interp(all_azi, cov_a[sort_idx], cov_r[sort_idx])

    if len(slice_rings) < 2:
        print("  Need at least 2 horizontal slices to create synthetic rings between them.")
        return pd.DataFrame()

    valid_depths = sorted(slice_rings.keys())
    ring_list    = []

    # Between each adjacent pair of real sections, add n_between synthetic rings
    for d0, d1 in zip(valid_depths[:-1], valid_depths[1:]):
        r0 = slice_rings[d0]
        r1 = slice_rings[d1]
        for t in np.linspace(0, 1, n_between + 2)[1:-1]:   # exclude the real sections
            d_syn = d0 + t * (d1 - d0)
            r_syn = (1 - t) * r0 + t * r1
            ring_list.append(pd.DataFrame({
                "azi":     all_azi,
                "depth":   d_syn,
                "r":       r_syn,
                "X0":      anchor_x,
                "Y0":      anchor_y,
                "Well_No": "cavern",
                "is_syn":  True,
            }))

    if not ring_list:
        return pd.DataFrame()

    out      = pd.concat(ring_list, ignore_index=True)
    out["x"] = out["X0"] + out["r"] * np.sin(np.deg2rad(out["azi"]))
    out["y"] = out["Y0"] + out["r"] * np.cos(np.deg2rad(out["azi"]))
    out["z"] = out["depth"]
    return out

print(f"\nBuilding Zone 2 rings ({Z_ZONE1_BOT} to {Z_ZONE2_BOT} ft)...")
syn_horiz = build_horiz_rings(
    horiz_real,
    depth_min = Z_ZONE2_BOT,
    depth_max = Z_ZONE1_BOT,
    dz        = dz,
    dazi      = dazi,
    anchor_x  = Xcav,
    anchor_y  = Ycav,
)
print(f"Zone 2 synthetic rings: {len(syn_horiz)} points")

# Merge all zones
syn = pd.concat([syn_vert, syn_horiz], ignore_index=True)
print(f"syn total (all zones): {len(syn)} points")

# --------------------------------------------------
# 7. Shape-preserving mirror for ppg4 in zones 1+3
# --------------------------------------------------
MIRROR_WELL      = "ppg4"
MIRROR_DEPTH_MIN = -2740
MIRROR_DEPTH_MAX = -2650

syn_mirror_list = []
for d in syn[syn["Well_No"] == MIRROR_WELL]["depth"].unique():
    if d < MIRROR_DEPTH_MIN or d > MIRROR_DEPTH_MAX:
        continue

    ring = syn[(syn["Well_No"] == MIRROR_WELL) & (syn["depth"] == d)]
    if ring.empty:
        continue

    covered_azi = ring["azi"].values
    covered_r   = ring["r"].values
    missing_azi = np.setdiff1d(np.arange(0, 360, dazi), covered_azi)

    if len(missing_azi) == 0:
        continue

    mirror_axis = find_mirror_axis(covered_azi)
    r_mirrored  = mirror_fill(covered_azi, covered_r, missing_azi, mirror_axis)

    X0 = ring["X0"].iloc[0]
    Y0 = ring["Y0"].iloc[0]

    syn_mirror_list.append(pd.DataFrame({
        "azi":     missing_azi,
        "depth":   d,
        "r":       r_mirrored,
        "X0":      X0,
        "Y0":      Y0,
        "Well_No": MIRROR_WELL,
        "is_syn":  True,
    }))

if syn_mirror_list:
    syn_mirror      = pd.concat(syn_mirror_list, ignore_index=True)
    syn_mirror["x"] = syn_mirror["X0"] + syn_mirror["r"] * np.sin(np.deg2rad(syn_mirror["azi"]))
    syn_mirror["y"] = syn_mirror["Y0"] + syn_mirror["r"] * np.cos(np.deg2rad(syn_mirror["azi"]))
    syn_mirror["z"] = syn_mirror["depth"]
    syn = pd.concat([syn, syn_mirror], ignore_index=True)
    print(f"Added {len(syn_mirror)} mirrored points for {MIRROR_WELL} "
          f"({MIRROR_DEPTH_MIN} to {MIRROR_DEPTH_MAX} ft)")

# --------------------------------------------------
# 8. Combine real + synthetic into GeoDataFrame
# --------------------------------------------------
vert_real   = gdf_edit[gdf_edit["orient"] == "v"].copy()
vert_coords = np.array([(g.x, g.y, g.z) for g in vert_real.geometry])

real_df = pd.DataFrame({
    "x":       [g.x for g in gdf_v.geometry],
    "y":       [g.y for g in gdf_v.geometry],
    "z":       [g.z for g in gdf_v.geometry],
    "azi":     gdf_v["azi"].values,
    "r":       gdf_v["r"].values,
    "depth":   gdf_v["depth"].values,
    "Well_No": gdf_v["Well_No"].values,
    "is_syn":  False,
})
real_df["depth"] = (real_df["depth"] / dz).round()   * dz
real_df["azi"]   = (real_df["azi"]   / dazi).round() * dazi % 360

# Combine: drop 'all' from real (vertical sections with cavern anchor),
# but keep 'cavern' from zone 2 synthetic.
combined = pd.concat([real_df[real_df["Well_No"] != "all"], syn], ignore_index=True)
combined = (combined.groupby(["Well_No", "depth", "azi"], as_index=False)
                    .agg({"x": "mean", "y": "mean", "z": "mean",
                          "r": "mean", "X0": "first", "Y0": "first",
                          "is_syn": "first"}))
combined = combined.sort_values(["Well_No", "depth", "azi"]).reset_index(drop=True)

combined["geometry"] = [Point(row.x, row.y, row.z) for row in combined.itertuples()]
gdf_combined = gpd.GeoDataFrame(combined, geometry="geometry", crs=gdf_edit.crs)
print(f"Combined GDF: {len(gdf_combined)} points")

plotter = pv.Plotter()
plotter.set_background("black")

# Synthetic -- colour by zone
if len(syn_vert) > 0:
    plotter.add_mesh(
        pv.PolyData(np.column_stack([syn_vert["x"], syn_vert["y"], syn_vert["z"]])),
        color="cyan", point_size=3, render_points_as_spheres=True,
        opacity=0.8, label="synthetic (vert. zones 1+3)"
    )
if len(syn_horiz) > 0:
    plotter.add_mesh(
        pv.PolyData(np.column_stack([syn_horiz["x"], syn_horiz["y"], syn_horiz["z"]])),
        color="lime", point_size=3, render_points_as_spheres=True,
        opacity=0.8, label="synthetic (horiz. zone 2)"
    )

# Real vertical points
plotter.add_mesh(
    pv.PolyData(vert_coords),
    color="yellow", point_size=5, render_points_as_spheres=True,
    opacity=0.6, label="real vertical"
)

# Real horizontal points
if len(horiz_real) > 0:
    horiz_coords = np.array([(g.x, g.y, g.z) for g in horiz_real.geometry])
    plotter.add_mesh(
        pv.PolyData(horiz_coords),
        color="red", point_size=5, render_points_as_spheres=True,
        opacity=0.6, label="real horizontal"
    )

plotter.add_legend(loc="upper right", size=(0.15, 0.1), bcolor="black")
plotter.add_axes()
plotter.show()


syn after r filter: 52319 points
Added 2394 mirrored points for ppg4 -2740 to -2650
Combined GDF: 56640 points


Widget(value='<iframe src="http://localhost:59490/index.html?ui=P_0x24e48a1e290_34&reconnect=auto" class="pyvi…

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 9.  Build 3-D surface mesh
# ─────────────────────────────────────────────────────────────────────────────
from scipy.interpolate import interp1d

CHUNK_COLORS = ["steelblue", "mediumseagreen", "tomato", "gold", "mediumpurple"]
UPSAMPLE     = 8

all_azis = np.arange(0, 360, dazi)   # full 360° azimuth grid

# One (x,y,z) per (depth, azi) — real beats synthetic
mesh_df = (combined
           .sort_values("is_syn")
           .groupby(["depth", "azi"], as_index=False)
           .first()
           .sort_values(["depth", "azi"]))

depths = np.sort(mesh_df["depth"].unique())

# Split at zone gaps so different zones are never bridged
gaps      = np.diff(depths)
split_idx = np.where(np.abs(gaps) > 1.5 * dz)[0] + 1
chunks    = np.split(depths, split_idx)

pl = pv.Plotter()
pl.set_background("black")

for i, depth_group in enumerate(chunks):
    if len(depth_group) < 2:
        continue

    sub = mesh_df[mesh_df["depth"].isin(depth_group)]

    # Pivot to the FULL azimuth ring (NaN where no data)
    Xp = (sub.pivot_table(index="depth", columns="azi", values="x", aggfunc="mean")
             .reindex(index=depth_group, columns=all_azis))
    Yp = (sub.pivot_table(index="depth", columns="azi", values="y", aggfunc="mean")
             .reindex(index=depth_group, columns=all_azis))

    X = Xp.values.copy()
    Y = Yp.values.copy()

    # Circular interpolation along azi for every depth row:
    # extend data ±360° so gaps wrap around the ring correctly
    for i_d in range(len(depth_group)):
        for arr in [X, Y]:
            row        = arr[i_d]
            known      = ~np.isnan(row)
            if known.sum() < 2:
                continue
            ka  = all_azis[known]
            kv  = row[known]
            ext_azi = np.r_[ka - 360, ka, ka + 360]
            ext_val = np.r_[kv,       kv, kv      ]
            arr[i_d] = np.interp(all_azis, ext_azi, ext_val)

    # Close ring (column 360 == column 0)
    X = np.column_stack([X, X[:, 0]])
    Y = np.column_stack([Y, Y[:, 0]])
    Z = np.tile(depth_group[:, None], (1, len(all_azis) + 1))

    # Fix any row still all-NaN
    for arr in [X, Y]:
        col_means = np.nanmean(arr, axis=0)
        nan_mask  = np.isnan(arr)
        arr[nan_mask] = np.take(col_means, np.where(nan_mask)[1])

    if np.isnan(X).any() or np.isnan(Y).any():
        continue

    # Upsample depth axis (cubic spline)
    n_fine   = (len(depth_group) - 1) * UPSAMPLE + 1
    fine_dep = np.linspace(depth_group[0], depth_group[-1], n_fine)
    kind     = "cubic" if len(depth_group) >= 4 else "linear"

    X_fine = np.stack([interp1d(depth_group, X[:, j], kind=kind)(fine_dep)
                       for j in range(X.shape[1])], axis=1)
    Y_fine = np.stack([interp1d(depth_group, Y[:, j], kind=kind)(fine_dep)
                       for j in range(X.shape[1])], axis=1)
    Z_fine = np.tile(fine_dep[:, None], (1, X.shape[1]))

    grid = pv.StructuredGrid(
        X_fine.T[:, :, np.newaxis],
        Y_fine.T[:, :, np.newaxis],
        Z_fine.T[:, :, np.newaxis],
    )
    pl.add_mesh(grid, color=CHUNK_COLORS[i % len(CHUNK_COLORS)],
                opacity=0.85, show_edges=False)

pl.add_axes()
pl.reset_camera()
pl.show()
